In [9]:
from pathlib import Path
import os
import sys
from dotenv import load_dotenv
load_dotenv()

project_root = Path.cwd().parent

os.chdir(project_root)

sys.path.insert(0, str(project_root / "src"))

from agent.graph import knowledgebase_search

result_no_filter = knowledgebase_search.invoke({
    "query": "leaf circular tan spots yellow halo rot symptoms treatment",
    "k": 3,
    "fetch_k": 15
})

result_with_filter = knowledgebase_search.invoke({
    "query": "leaf circular tan spots yellow halo rot symptoms treatment",
    "plant": "grape",
    "k": 3,
    "fetch_k": 15
})

def print_retrieved_docs(docs):
    print(f"--- {len(docs)} dokumen ditemukan ---")
    for i, doc in enumerate(docs, 1):
        if isinstance(doc, dict):
            content = doc.get('page_content', '').strip()
            meta = doc.get('metadata', {})
        else:
            content = getattr(doc, 'page_content', '').strip()
            meta = getattr(doc, 'metadata', {})
            
        score = meta.get('relevance_score')
        score_str = f"{score:.4f}" if score is not None else "T/A"
        
        print(f"[{i}] Skor: {score_str} | Tanaman: {meta.get('plant', '-')} | Penyakit: {meta.get('disease', '-')} | Kategori: {meta.get('category', '-')}")
        print(f"Konten: {content}")
        print("-" * 50)

if 'result_no_filter' in locals():
    print("\nHasil dari 'result_no_filter':")
    print_retrieved_docs(result_no_filter)

if 'result_with_filter' in locals():
    print("\nHasil dari 'result_with_filter':")
    print_retrieved_docs(result_with_filter)


Hasil dari 'result_no_filter':
--- 3 dokumen ditemukan ---
[1] Skor: 0.6914 | Tanaman: Wheat | Penyakit: Tan Spot Pyrenophora Tritici-repentis | Kategori: fungal
Konten: Disease: Tan Spot Pyrenophora Tritici-repentis affecting Wheat

Symptoms: Oval or diamond shaped necrotic lesions with brown centers and yellow halos on leaves. Management: Disease can be significantly reduced by rotating crops with non-hosts and tilling crop debris into soil after harvest
--------------------------------------------------
[2] Skor: 0.6758 | Tanaman: Almond | Penyakit: Alternaria Leaf Spot | Kategori: fungal
Konten: Disease: Alternaria Leaf Spot affecting Almond

Symptoms: Light brown lesions on leaves which expand to form circular lesions on leaf blade or semi-circular lesions on margin; leaves may develop light yellow necrosis which dries and turns tan in center of leaves; infected leaves dropping from tree; fruit does not drop from tree. Management: Late spring treatment with appropriate fungicide 

In [10]:
from pathlib import Path
import os
import sys
from dotenv import load_dotenv
load_dotenv()

try:
    project_root = Path(__file__).parents[1]
except NameError:
    project_root = Path.cwd()
    if project_root.name == 'notebooks':
        project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from agent.graph import web_search

def print_web_results(docs):
    print(f"--- {len(docs)} hasil web ditemukan ---")
    for i, doc in enumerate(docs, 1):
        if isinstance(doc, dict):
            content = doc.get('page_content', '').strip()
            meta = doc.get('metadata', {})
        else:
            content = getattr(doc, 'page_content', '').strip()
            meta = getattr(doc, 'metadata', {})
            
        print(f"[{i}] Judul: {meta.get('title', 'Tanpa Judul')}")
        print(f"    URL: {meta.get('url', 'T/A')}")
        print(f"    Konten: {content}")
        print("-" * 50)

query = "current treatment for tomato early blight"
print(f"Mencari: {query}...\n")

web_results = web_search.invoke({"query": query})

print_web_results(web_results)

Mencari: current treatment for tomato early blight...

--- 3 hasil web ditemukan ---
[1] Judul: Early/Late Tomato Blight | Thuss Greenhouses
    URL: https://thussgreenhouses.com/tomato-blight/
    Konten: Early blight can be treated by removing the infected leaves and by spraying with copper or sulfur. Importantly, you can harvest the fruit from tomatoes with
--------------------------------------------------
[2] Judul: What treatment can I use for early blight on my tomatoes? - Facebook
    URL: https://www.facebook.com/groups/1156935537680311/posts/26266090616338123/
    Konten: Copper-based fungicides, Chlorothalonil, or Mancozeb are effective against late blight. Apply the fungicide according to the label instructions,
--------------------------------------------------
[3] Judul: How to Control Early Blight of Tomatoes and Potatoes
    URL: https://www.almanac.com/pest/early-blight
    Konten: Treatments for Early Blight · Keep your plants growing vigorously · Irrigate from below 

In [ ]:
from agent.graph import plant_disease_identification, closed_set_leaf_detection
import asyncio

# Mock Runtime untuk mensimulasikan environment agent
class MockRuntime:
    def __init__(self, state):
        self.state = state
        self.tool_call_id = "test_call_id"

# Setup variabel dan fungsi helper
image_url = "https://preview.redd.it/help-my-grape-vine-seems-to-be-sick-v0-g619rhhxz61b1.jpg?width=1080&crop=smart&auto=webp&s=9469c8202f5c7a4d55c8c7422cf61538bdc4bc67"

def print_tool_result(result, title="Hasil Tool"):
    print(f"\n=== {title} ===")
    if hasattr(result, 'update') and 'messages' in result.update:
        messages = result.update['messages']
        for msg in messages:
            if hasattr(msg, 'content'):
                content = msg.content
                if isinstance(content, list):
                    for item in content:
                        if isinstance(item, dict):
                            if item.get('type') == 'text':
                                print(item.get('text', ''))
                            elif item.get('type') == 'image_url':
                                print(f"[Gambar Visualisasi: {item.get('image_url', '')}]")
                else:
                    print(content)
    else:
        print(result)

# Skenario 1: Identifikasi Tanpa Deteksi (Full Image)
print("--- Memulai Skenario 1: Identifikasi Tanpa Deteksi ---")

state_no_detection = {
    "current_image_url": image_url,
    "detections": []
}

runtime_no_det = MockRuntime(state_no_detection)

# Invoke tool secara langsung (bypass agent)
try:
    result_no_det = await plant_disease_identification.coroutine(
        query_text="grape leaf circular tan spots with dark border and black specks black rot",
        use_detections=False,
        runtime=runtime_no_det
    )
    print_tool_result(result_no_det, "Identifikasi Tanpa Deteksi")
except Exception as e:
    print(f"Error executing tool: {e}")
    

--- Memulai Skenario 1: Identifikasi Tanpa Deteksi ---

=== Identifikasi Tanpa Deteksi ===
Full-Image Classification (text-to-image)
Query: 'grape leaf circular tan spots with dark border and black specks black rot'
Reranking: Enabled

Predicted Label: grape black rot
Confidence: 0.5656

Label Scores:
  grape downy mildew: 0.5740
  grape black rot: 0.5656
  grape leaf spot: 0.5529

Top-5 Results:
  1. grape downy mildew (0.5740)
     Plant: grape
     Caption: Grape downy mildew appears as fuzzy grayish-white patches on the leaves, stems, ...
     Image: https://thesis-assets.andyathsid.com/plantwild/gallery/images/grape%20downy%20mildew/google_0255.jpg
  2. grape black rot (0.5659)
     Plant: grape
     Caption: Grape black rot appears as small, dark brown to black circular spots on the frui...
     Image: https://thesis-assets.andyathsid.com/plantwild/gallery/images/grape%20black%20rot/google_0341.jpg
  3. grape black rot (0.5653)
     Plant: grape
     Caption: Grape black rot appe

In [ ]:

# Skenario 2: Identifikasi Dengan Deteksi (Region Based)
print("\n--- Memulai Skenario 2: Identifikasi Dengan Deteksi ---")

state_with_detection = {
    "current_image_url": image_url,
    "detections": []
}
runtime_det = MockRuntime(state_with_detection)

# Langkah 1: Jalankan Deteksi Daun (Closed Set)
print("1. Menjalankan deteksi daun...")
try:
    detection_result = await closed_set_leaf_detection.coroutine(
        confidence_threshold=0.3,
        runtime=runtime_det
    )
    
    print_tool_result(detection_result, "Hasil Deteksi Objek")
    
    # Update state dengan hasil deteksi
    if hasattr(detection_result, 'update') and 'detections' in detection_result.update:
        detections = detection_result.update['detections']
        state_with_detection['detections'] = detections
        print(f"\n--> Sukses! {len(detections)} deteksi ditambahkan ke state.")
    else:
        print("\n--> Peringatan: Tidak ada deteksi yang dikembalikan.")

    # Langkah 2: Jalankan Identifikasi Penyakit dengan data deteksi
    print("\n2. Menjalankan identifikasi penyakit pada region terdeteksi...")
    result_with_det = await plant_disease_identification.coroutine(
        query_text="grape leaf circular tan spots with dark border and black specks black rot",
        use_detections=True,
        runtime=runtime_det
    )
    
    print_tool_result(result_with_det, "Identifikasi Dengan Deteksi")

except Exception as e:
    print(f"Error executing scenario 2: {e}")


--- Memulai Skenario 2: Identifikasi Dengan Deteksi ---
1. Menjalankan deteksi daun...

=== Hasil Deteksi Objek ===
Detection Summary:
0: 2 detection(s)

Detailed Detections:
0: 0.802 at [206.9, 396.9, 819.9, 980.4]
0: 0.329 at [334.0, 82.0, 809.7, 510.8]

[Gambar Visualisasi: https://thesis-assets.andyathsid.com/detection-results/detection_c43cd025-0869-4ef9-8e2b-4c8e5288126b.png]

--> Sukses! 2 deteksi ditambahkan ke state.

2. Menjalankan identifikasi penyakit pada region terdeteksi...

=== Identifikasi Dengan Deteksi ===
Region-Based Classification (text-to-image)
Analyzed 2 detected regions
Reranking: Enabled

Region 1 [206.9, 396.9, 819.9, 980.4]:
  Label: grape black rot (confidence: 0.5656)
  Top predictions:
    1. grape downy mildew (0.574) - https://thesis-assets.andyathsid.com/plantwild/gallery/images/grape%20downy%20mildew/google_0255.jpg
    2. grape black rot (0.566) - https://thesis-assets.andyathsid.com/plantwild/gallery/images/grape%20black%20rot/google_0341.jpg
    

: 